In [ ]:
import os
import textwrap
import threading
import torch
import csv
import os
from unsloth import FastLanguageModel
from PIL import Image
from unsloth.chat_templates import get_chat_template
from qiskit import QuantumCircuit
from qiskit.visualization import circuit_drawer
# from src.utils import find_highest_checkpoint
# from transformers import TextIteratorStreamer

# Globals for holding the loaded model and processor
MODEL = None
TOKENIZER = None


def initialize_model(model_id: str, checkpoint_root: str = "./model_cp"):
    global MODEL, TOKENIZER

    # If already loaded, just return
    if MODEL is not None and TOKENIZER is not None:
        return MODEL, TOKENIZER

    # Check if local fine-tuned model is present and non-empty
    # try:
    #     adapter_path = find_highest_checkpoint(checkpoint_root)
    #     print(f"Highest checkpoint found: {adapter_path}")
    #     model_name = adapter_path
    # except:
    model_name = model_id

    print(f"Loading model from: {model_name}")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_name,
        load_in_4bit=True,
    )
    
    MODEL = model
    TOKENIZER = tokenizer
    return MODEL, TOKENIZER


def format_data_inference(tokenizer, user_input, model_id: str) -> str:
    template_name = None
    model_id_lower = model_id.lower()

    if "mistral" in model_id_lower:
        template_name = "mistral"
    elif "llama" in model_id_lower:
        template_name = "llama-3"
    elif "qwen" in model_id_lower:
        template_name = "qwen2.5"
    # elif "deepseek" in model_id_lower and "qwen" in model_id_lower:
    #     template_name = None

    if template_name:
        row_json = [{"role": "user", "content": user_input}]
        tokn = get_chat_template(
            tokenizer,
            chat_template=template_name,
            mapping={"role": "from", "content": "value", "user": "human", "assistant": "gpt"},
            map_eos_token=True,
        )
        try:
            formatted_text = tokn.apply_chat_template(
                row_json,
                tokenize=False,
                add_generation_prompt=False
            )
        except Exception:
            formatted_text = f"### Instruction:\n{user_input}\n### Response:\n"
    elif "deepseek" in model_id_lower and "qwen" in model_id_lower:
        formatted_text = f"### Instruction:\n{user_input}\n### Response:\n"
    else:
        formatted_text = (
            f"<|im_start|>user\n{user_input}<|im_end|>\n"
            f"<|im_start|>assistant\n"
        )

    return formatted_text


def run_inference_lm(user_input: str, temperature: float = 1.0, max_tokens: int = 1500, model_id: str = "unsloth/Phi-3.5-mini-instruct") -> str:
    model, tokenizer = initialize_model(model_id)
    FastLanguageModel.for_inference(model)
    prompt = format_data_inference(tokenizer, user_input, model_id) 

    # 4. Tokenize inputs
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False,
    )
    inputs = {k: v.to("cuda") for k, v in inputs.items()}

    # 5. Generate response
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        temperature=temperature,
        # pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        repetition_penalty=1.1,
        use_cache=True 
    )
    generated_text = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:], 
        skip_special_tokens=True
    )
    if "llama" in model_id.lower():
        unwanted_prefix = "assistant\n\n"
        if generated_text.startswith(unwanted_prefix):
            generated_text = generated_text[len(unwanted_prefix):].lstrip()
    
    return generated_text

def get_prompt(qubit, depth):
    return textwrap.dedent(f"""\
DO NOT SUMMARIZE. ONLY OUTPUT RAW TAGGED BLOCKS.
Generate random QPE circuit with depth {depth} and number of qubits {qubit} (Qiskit qasm), using logical gate, use only minimal classical register for measurement
For the circuit, do the following: 

Start with a reasoning block in this format (EXACTLY FOLLOW THIS FORMAT WITH NUMBERING):
<thought>  
1. Based on the image, consist of pattern .. there is CCNOT and X gate as oracle then it is classified as Grover (as detail as possible) (Might be different for QPE, QML etc) 
(You may add more steps here as detail as possible)
(as detail as possible, you may add more here EXAMPLE: (not strictly following this)
- Based on the imagined image, the circuit starts with Hadamard gates on both qubits. This suggests a superposition state.
- The oracle appears to use X gates followed by CZ, then X gates again to mark a specific state like |11⟩ or |01⟩.
- The circuit uses 2 qubits: q[0], q[1].
- Classical registers are assumed: c[0], c[1].
- Total circuit depth is approximately 6:
   * Layer 1: Hadamard
   * Layer 2: X (pre-oracle)
   * Layer 3: CZ gate
   * Layer 4: Undo X
   * Layer 5: Diffusion
   * Layer 6: Measurement
- Oracle targets a specific basis state.
- No learned parameters or embeddings used.
- Logical gate layout is inferred visually.
)

2. Number of qubits, how many quantum classical registers, is there measurement, reverse engineering from image to thinking process, how to classify number of qubits, registers, and measurement

3. Composition - Logical GATE (reverse OCR) (The ";" represents depth, if 7 depths means 7 times ";"): 
eg: 
8-depth and 7-qubit:
rz0 none none none none none none; sx0 none none none none none none; rz0 none none none none none none; sx0 none none none none none none; rz0 none none none none none none; rz0 none none none none none none; x0 none none none none none none; measure0 none none none none none none
4-depth and 7-qubit:
rz0 none none none none none none; rz0 none none none none none none; rz0 none none none none none none; measure0 none none none none none none

4. Code comments step by step
// Initialization, // Oracle for |11>, // Diffusion, // Measurement
</thought>

Then output the full OpenQASM 2.0 code block in this format:
<OPENQASM code>
OPENQASM 2.0;
include "qelib1.inc";

qreg q[2];
creg c[2];

// Initialization
h q[0];
h q[1];

// Oracle for |11>
x q[0];
x q[1];
cz q[0], q[1];
x q[0];
x q[1];

// Diffusion
h q[0];
h q[1];
x q[0];
x q[1];
cz q[0], q[1];
x q[0];
x q[1];
h q[0];
h q[1];

// Measurement
measure q[0] -> c[0];
measure q[1] -> c[1];
</OPENQASM code>

Output only start with <thought> .. </thought> and end with <OPENQASM code> ... </OPENQASM code> sections for the circuit. 
Do not explain, summarize, or add commentary outside these tags. also must include 'include "qelib1.inc";' after OPENQASM 2.0; inside <OPENQASM code> ... </OPENQASM code> section.
Make sure the open qasm code is working and there is no wrong syntax like needed the end of argument list error (example: for(i = 0; i < 7; ++i) h q[i]; => please run this code to generate the right syntax). make sure the code is open qasm 2.0 compatible and not other code.

IMPORTANT:
- DO NOT use any for-loops. Write out every gate explicitly, one per line.
- DO NOT use any gates that are not supported by OpenQASM 2.0, such as `cu1`, `u1`, `u2`, or `u3`. Only use gates from "qelib1.inc" like h, x, cx, cz, rz, sx, and measure.
- DO NOT write pseudocode. ONLY produce real, working OpenQASM 2.0 syntax that can run in Qiskit or IBM Q runner without modification.
- All code must be fully expanded. No loops, no macros, no `for`, `while`, or similar.
""")
    
def extract_qasm_block(response: str) -> str:
    start = response.find("<OPENQASM code>")
    end = response.find("</OPENQASM code>")
    if start == -1 or end == -1:
        raise ValueError("Missing <OPENQASM code> block in response")
    return response[start + len("<OPENQASM code>"):end].strip()


model_id = "unsloth/Qwen2.5-14B-Instruct-unsloth-bnb-4bit"
folder = "Images/QPE"
trial = 5

# number of qubits (1-10)
# circuit_depth (2-8)

csv_path = os.path.join(folder, "qpe_results.csv")
if not os.path.exists(csv_path):
    with open(csv_path, mode="w", newline='') as f:
        writer = csv.writer(f)
        writer.writerow(["image", "type", "qubits", "depth", "response"])

def variation(qubits, depth):
    return round(
        4 + (100 - 4) *
        ((qubits - 2) / 6) ** 0.6 *
        (depth / 8) ** 2.2
    )

for n_qubit in range(2, 11):
    for n_depth in range(2, 9):
        num_variation = variation(n_qubit, n_depth)
        for var in range(num_variation):
            for trial_number in range(trial):
                try:
                    response = run_inference_lm(get_prompt(n_qubit, n_depth), model_id=model_id, max_tokens=4500, temperature=0.1)
                    response_qasm = extract_qasm_block(response)
                    qc = QuantumCircuit.from_qasm_str(response_qasm)
            
                    # 3. Draw and save the circuit image
                    fig = circuit_drawer(qc, output='mpl')
                    img_path = f"{folder}/QPE_{n_qubit}_{n_depth}_{var}.png"
                    fig.savefig(img_path)

                    # Save metadata to CSV
                    with open(csv_path, mode="a", newline='') as f:
                        writer = csv.writer(f)
                        writer.writerow([img_path, "QPE", n_qubit, n_depth, response])
                    break
                except:
                    print(f'Failed, trial {trial_number}')


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/cqilab/anaconda3/envs/llmfinetune/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-07-02 18:53:30.451784: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-07-02 18:53:30.466561: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1751450010.483274 1398472 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1751450010.488097 1398472 cuda_blas

[2025-07-02 18:53:33,119] [INFO] [real_accelerator.py:239:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/home/cqilab/anaconda3/envs/llmfinetune/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/home/cqilab/anaconda3/envs/llmfinetune/compiler_compat/ld: cannot find -lcufile: No such file or directory
collect2: error: ld returned 1 exit status


🦥 Unsloth Zoo will now patch everything to make training faster!
Loading model from: unsloth/Qwen2.5-14B-Instruct-unsloth-bnb-4bit
==((====))==  Unsloth 2025.6.1: Fast Qwen2 patching. Transformers: 4.52.0.dev0.
   \\   /|    NVIDIA GeForce RTX 3090. Num GPUs = 2. Max memory: 23.684 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu124. CUDA: 8.6. CUDA Toolkit: 12.4. Triton: 3.1.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post1. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards: 100%|██████████| 3/3 [00:02<00:00,  1.07it/s]


Failed, trial 0
Failed, trial 1
Failed, trial 2
Failed, trial 3
Failed, trial 4
Failed, trial 0
Failed, trial 1
Failed, trial 2
Failed, trial 3
Failed, trial 4
Failed, trial 0
Failed, trial 1
Failed, trial 2
Failed, trial 3
Failed, trial 4
Failed, trial 0
Failed, trial 1
Failed, trial 2
Failed, trial 3
Failed, trial 4
